# MERT vs CultureMERT: 30-Second Raga Benchmark

This notebook compares frozen MERT-95M and CultureMERT-95M representations on six Saraga Carnatic ragas.

**Experiment:** six tracks per raga, track-wise 4/1/1 train-validation-test split, four 30-second clips per track, all-layer linear probes, and a small external neural classifier on the selected layer.

The original MERT models are not updated. Only the classifiers are trained. Expected runtime is roughly 2-4 hours after the dataset is available.

## 1. Check Kaggle settings

Before running, open **Notebook options** and enable a **GPU accelerator** and **Internet**.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

import torch

assert Path('/kaggle/working').exists(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), (
    'GPU is not enabled. Open Notebook options and choose a T4 or P100 accelerator.'
)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

try:
    urllib.request.urlopen('https://huggingface.co', timeout=15)
    print('Internet: available')
except Exception as exc:
    raise RuntimeError(
        'Internet is disabled. Enable Internet in Kaggle Notebook options and rerun.'
    ) from exc

## 2. Attach Saraga as a Kaggle Input

The full Zenodo archive is too large for Kaggle's working disk. Click **Add Input**, search for **Saraga Carnatic Music Dataset** by `desolationofsmaug`, and add it to the notebook:

https://www.kaggle.com/datasets/desolationofsmaug/saraga-carnatic-music-dataset

The next cell finds the attached read-only dataset automatically.

In [ ]:
import shutil

OUTPUT_DIR = Path('/kaggle/working/mert_raga_30s')
candidates = [
    path for path in Path('/kaggle/input').rglob('carnatic')
    if any(path.rglob('*.mp3')) and any(path.rglob('*.json'))
]
assert candidates, (
    "Saraga input not found. Click Add Input and attach "
    "'Saraga Carnatic Music Dataset' by desolationofsmaug."
)
KAGGLE_DATA_ROOT = candidates[0]

# Remove any large incomplete Zenodo download from an earlier attempt.
failed_download = Path('/kaggle/working/saraga_carnatic')
if failed_download.exists():
    shutil.rmtree(failed_download)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Attached Saraga directory:', KAGGLE_DATA_ROOT)
print('Results directory:', OUTPUT_DIR)

## 3. Install packages and get the project

Transformers is pinned because MERT uses custom Hugging Face code. Warnings printed by `pip` can usually be ignored unless the cell ends with an error.

In [ ]:
!pip install -q "transformers==4.41.0" "librosa>=0.10" soundfile pandas scikit-learn matplotlib seaborn tqdm mirdata joblib nnAudio

import subprocess

REPO_DIR = Path('/kaggle/working/mert-raga-classification')
REPO_URL = 'https://github.com/propixx/mert-raga-classification.git'
REPO_BRANCH = 'codex/kaggle-30s-benchmark'
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('Project directory:', Path.cwd())
subprocess.run([sys.executable, 'scripts/00_check_env.py'], check=True)

## 4. Run the complete benchmark

This is the long cell. Completed clips and each model/split embedding file are cached with a configuration fingerprint. If the run is interrupted, run the notebook again and this cell will reuse completed work.

The first run reads Saraga from the attached Kaggle input and downloads only the two foundation models. Do not use `--force-clips` or `--force-embeddings` unless you deliberately want to rebuild the caches. If Kaggle reports a GPU out-of-memory error, change the batch-size value below from `2` to `1`; completed split caches will still be reused.

In [ ]:
command = [
    sys.executable,
    'scripts/10_balanced_benchmark.py',
    '--dataset', 'saraga_carnatic',
    '--kaggle-data-root', str(KAGGLE_DATA_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--num-ragas', '6',
    '--tracks-per-raga', '6',
    '--segments-per-track', '4',
    '--segment-seconds', '30',
    '--batch-size', '2',
    '--head-hidden-dim', '256',
    '--head-dropout', '0.2',
    '--head-lr', '0.001',
    '--head-epochs', '100',
    '--head-patience', '12',
]

print('Running:', ' '.join(command))
subprocess.run(command, check=True)

## 5. Read the results

The report compares the linear probe and external neural head for both foundation models. Macro F1 is especially useful because it gives equal importance to every raga.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

report_path = OUTPUT_DIR / 'REPORT.md'
summary_path = OUTPUT_DIR / 'metrics' / 'benchmark_summary.csv'
assert report_path.exists(), 'The benchmark did not finish; inspect the previous cell.'

display(Markdown(report_path.read_text(encoding='utf-8')))
display(pd.read_csv(summary_path))

figure_names = [
    'dataset_distribution.png',
    'mert_95m_layer_curve.png',
    'culturemert_95m_layer_curve.png',
    'mert_95m_linear_confusion_matrix.png',
    'culturemert_95m_linear_confusion_matrix.png',
    'mert_95m_external_head_training.png',
    'culturemert_95m_external_head_training.png',
]
for name in figure_names:
    path = OUTPUT_DIR / 'figures' / name
    if path.exists():
        display(Image(filename=str(path), width=850))

## 6. Save the submission package

The ZIP contains the report, metrics, plots, split details, linear probes, and neural classifier checkpoints. It excludes the large Saraga audio and embedding cache.

Use **Save Version** in Kaggle after the notebook finishes. The ZIP will also appear in the notebook Output files.

In [ ]:
from IPython.display import FileLink, display

result_zip = OUTPUT_DIR / 'mert_raga_30s_results.zip'
assert result_zip.exists(), f'Result ZIP not found: {result_zip}'
print('Result package:', result_zip)
print('Size: %.2f MB' % (result_zip.stat().st_size / 1e6))
display(FileLink(str(result_zip)))